# Đặt tên Col cho 3 files data

In [2]:
rating = r'F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\ratings.dat'
movies = r'F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\movies.dat'
users = r'F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\users.dat'

In [3]:
import pandas as pd
import numpy as np

# 1. Đọc file ratings (Quan trọng nhất)
ratings_cols = ['UserID', 'MovieID', 'Rating', 'Timestamp']
ratings = pd.read_csv(rating, sep='::', engine='python', names=ratings_cols)

# 2. Đọc file movies và users (Dùng để lấy thêm tính năng nếu cần)
movies_cols = ['MovieID', 'Title', 'Genres']
movies = pd.read_csv(movies, sep='::', engine='python', names=movies_cols, encoding='latin-1')

users_cols = ['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code']
users = pd.read_csv(users, sep='::', engine='python', names=users_cols)

# Sắp xếp lịch sử đánh giá của từng user theo thời gian (Timestamp) tăng dần
ratings = ratings.sort_values(by=['UserID', 'Timestamp']).reset_index(drop=True)

In [4]:
def check_dataframe_info(df, name):
    print(f"=========================================")
    print(f"THÔNG TIN BẢNG: {name.upper()}")
    print(f"=========================================")
    # df.shape trả về một tuple: (số dòng, số cột)
    print(f"1. Số lượng mẫu (Dòng): {df.shape[0]:,}")
    print(f"2. Số lượng đặc trưng (Cột): {df.shape[1]}")
    print(f"3. Kiểu dữ liệu của từng cột:")
    # df.dtypes in ra kiểu dữ liệu của tất cả các cột
    print(df.dtypes)
    print("\n")

# Chạy hàm kiểm tra cho từng DataFrame
check_dataframe_info(ratings, "Ratings")
check_dataframe_info(users, "Users")
check_dataframe_info(movies, "Movies")

THÔNG TIN BẢNG: RATINGS
1. Số lượng mẫu (Dòng): 1,000,209
2. Số lượng đặc trưng (Cột): 4
3. Kiểu dữ liệu của từng cột:
UserID       int64
MovieID      int64
Rating       int64
Timestamp    int64
dtype: object


THÔNG TIN BẢNG: USERS
1. Số lượng mẫu (Dòng): 6,040
2. Số lượng đặc trưng (Cột): 5
3. Kiểu dữ liệu của từng cột:
UserID         int64
Gender        object
Age            int64
Occupation     int64
Zip-code      object
dtype: object


THÔNG TIN BẢNG: MOVIES
1. Số lượng mẫu (Dòng): 3,883
2. Số lượng đặc trưng (Cột): 3
3. Kiểu dữ liệu của từng cột:
MovieID     int64
Title      object
Genres     object
dtype: object




# Mã hóa ID và định nghĩa MASK/PAD

Số 0: Dành riêng làm khoảng trống ([PAD]) để bù vào các chuỗi phim bị ngắn.

Số 1 đến 3706: Dành cho các bộ phim có thật (đáp án).

Số 3707 (len + 1): Dành riêng làm Token che giấu ([MASK]), báo hiệu cho AI biết đây là "chỗ trống cần suy luận".


Tạo số 3707(blank) để nó che các phim trong chuỗi lại ví dụ

Giả sử ta chọn số 2 (phim Jumanji) làm ký hiệu để che.

-Chuỗi gốc người dùng xem: [Toy Story (1), Heat (6), Casino (16)]

-Hành động: Bạn muốn kiểm tra mô hình bằng cách giấu phim Heat (6) đi.

-Nếu bạn dán số 2 đè lên số 6: Mảng dữ liệu biến thành [1, 2, 16].

->Hậu quả: Khi máy tính đọc mảng [1, 2, 16], nó sẽ hiểu là: "À, người dùng này đã xem Toy Story, sau đó xem Jumanji, rồi xem Casino. Một chuỗi rất bình thường, chẳng có câu hỏi nào cần giải quyết ở đây cả!". Mô hình hoàn toàn không biết rằng vị trí ở giữa đã bị thay đổi và cần phải được dự đoán. Nó đã bị "đánh lừa" thành một chuỗi hợp lệ khác.

Giải pháp:

Thay vì dùng số 2, ta dùng một con số nằm ngoài danh sách phim có thật là 3707.

Chuỗi gốc: [Toy Story (1), Heat (6), Casino (16)]

Hành động: Bạn dán miếng băng keo 3707 đè lên phim Heat (6).

Kết quả: Mảng dữ liệu biến thành [1, 3707, 16].

Máy tính sẽ hiểu: "Khoan đã! Từ điển phim chỉ có từ 1 đến 3706 thôi. Số 3707 này là một ký hiệu đặc biệt báo hiệu chỗ trống (Fill-in-the-blank). Nhiệm vụ của mình là phải nhìn vào phim số 1 và phim số 16 ở hai bên, để suy luận xem cái phim nằm dưới lớp băng keo 3707 kia thực chất là phim số mấy trong tập từ 1 đến 3706!"

In [5]:
# Tạo từ điển mã hóa MovieID thành các số nguyên liên tục bắt đầu từ 1
raw_movie_ids = sorted(ratings['MovieID'].unique())
movie2id = {raw_id: i + 1 for i, raw_id in enumerate(raw_movie_ids)}

# Số lượng phim thực tế
num_items = len(movie2id)

# Định nghĩa mã cho token [MASK]
MASK_TOKEN = num_items + 1  # Token này nằm ngay sau ID phim cuối cùng

# Áp dụng mã hóa vào bảng ratings
ratings['Movie_Encoded'] = ratings['MovieID'].map(movie2id)

# Gom nhóm thành chuỗi lịch sử (Sequence Generation)

Gom các đánh giá rời rạc thành các chuỗi hành vi (sequences) của từng người dùng

In [6]:
user_sequences = ratings.groupby('UserID')['Movie_Encoded'].apply(list).to_dict()

**Output mẫu**

{

    1: [32, 15, 114, 5, ...], # Lịch sử xem của UserID 1
    
    2: [18, 5, 233, ...],     # Lịch sử xem của UserID 2
    ...
}

In [7]:
print(user_sequences[1])  # In ra lịch sử đánh giá của user có ID = 1 sau khi mã hóa

[2970, 1179, 1575, 958, 2148, 1659, 3178, 2600, 1118, 1105, 690, 254, 859, 594, 2489, 1782, 1849, 2890, 878, 971, 1783, 1839, 145, 964, 1026, 854, 1196, 2593, 2558, 1155, 640, 2711, 518, 2899, 2587, 2129, 965, 1108, 581, 2206, 1422, 514, 582, 2484, 709, 575, 1, 2163, 2103, 741, 1440, 1728, 48]


In [8]:
print(user_sequences[2]) 

[1109, 1121, 1128, 2513, 1202, 2736, 1136, 1105, 310, 2817, 2652, 1124, 1766, 1118, 580, 2880, 3236, 1694, 502, 1019, 2308, 2822, 107, 1887, 2932, 1156, 2890, 1260, 1107, 1778, 1774, 860, 1657, 1013, 1783, 3239, 3413, 3494, 1168, 1775, 1619, 2524, 1789, 1032, 842, 3220, 3342, 2646, 3108, 2854, 259, 2121, 577, 1162, 2857, 1153, 3458, 1776, 1154, 2047, 3437, 921, 2014, 2079, 1338, 3032, 627, 229, 1025, 1048, 485, 1155, 3648, 1415, 1100, 2204, 2167, 2129, 347, 2893, 1174, 3567, 576, 1849, 2375, 444, 2709, 1479, 467, 158, 371, 3187, 3033, 1307, 21, 340, 1407, 2161, 1827, 2087, 1272, 628, 2235, 1624, 1274, 1429, 2297, 1287, 738, 2675, 2892, 359, 1632, 160, 446, 429, 1467, 2427, 1554, 3034, 703, 1823, 1946, 284, 93, 1551, 421, 1421, 1738]


# Chia dữ liệu và Xử lý độ dài chuỗi

In [9]:
max_len = 50  # Độ dài chuỗi quy định cho mô hình

def pad_and_truncate(sequence, max_len):
    # Nếu chuỗi dài hơn max_len, cắt lấy các phần tử cuối cùng (gần nhất)
    if len(sequence) > max_len:
        return sequence[-max_len:]
    # Nếu ngắn hơn, thêm số 0 ([pad]) vào bên trái
    else:
        return [0] * (max_len - len(sequence)) + sequence

# Tạo dữ liệu mẫu cho một lượt huấn luyện (Train)
train_data = []
for user, seq in user_sequences.items():
    if len(seq) < 3: # Bỏ qua nếu user xem quá ít phim không đủ chia train/val/test
        continue
        
    # Lấy chuỗi học (bỏ phim cuối và áp chót dành cho test/val)
    train_seq = seq[:-2]
    
    # Thực hiện Masking ngẫu nhiên 15% cho tập Train
    masked_seq = []
    labels = [] # Nhãn để tính Loss
    for item in train_seq:
        prob = np.random.rand()
        if prob < 0.15: # 15% cơ hội bị che
            masked_seq.append(MASK_TOKEN)
            labels.append(item) # Nhãn là bộ phim gốc
        else:
            masked_seq.append(item)
            labels.append(0) # 0 nghĩa là không tính loss cho vị trí này
            
    # Áp dụng Padding để đưa về kích thước cố định (50,)
    padded_input = pad_and_truncate(masked_seq, max_len)
    padded_labels = pad_and_truncate(labels, max_len)
    
    train_data.append((padded_input, padded_labels))

# BERT4REC